# MUSIN-G Dataset Loader Demo

This notebook demonstrates how to use the BCMIMusingLoader to load the MUSIN-G dataset,
which contains EEG recordings from 20 subjects listening to 12 different songs.

## Dataset Overview
- **Subjects**: 20
- **Songs**: 12 (various genres)
- **Sessions**: 12 per subject (one per song)
- **Sampling Rate**: 250 Hz
- **Format**: BIDS-compliant EEGLAB .set files
- **Music IDs**: Integer from 0 to 11 (MusingMusicId)

In [1]:
from pathlib import Path
import sys

# Add parent directory to path if needed
project_root = Path.cwd()
# if str(project_root) not in sys.path:
#     sys.path.insert(0, str(project_root))

from eeg_music.bcmi import BCMIMusingLoader, create_bcmi_loader
from eeg_music.data import MusingMusicId, MusicFilename, copy_from_dataloader_into_dir
from eeg_music.data import EEGMusicDataset

/home/zmrocze/studia/uwr/eeg-magisterka/.devenv/state/venv/lib/python3.12/site-packages/speechbrain/utils/torch_audio_backend.py:57: UserWarning: torchaudio._backend.list_audio_backends has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  available_backends = torchaudio.list_audio_backends()


## 1. Load MUSIN-G Dataset with BCMIMusingLoader

We can use either the specific loader class or the factory function.

In [2]:
# Path to the musin-g dataset
musin_g_path = Path(".") / "datasets" / "musin_g_data"

# Method 1: Direct instantiation
loader = BCMIMusingLoader(str(musin_g_path))

# Method 2: Using factory function (automatically detects dataset type)
# loader = create_bcmi_loader(str(musin_g_path))

print(f"Dataset: {loader.dataset_name}")
print(f"Number of subjects: {len(loader.subjects)}")
print(f"Subjects: {loader.subjects[:5]}...")  # Show first 5

Dataset: musin-g
Number of subjects: 20
Subjects: ['001', '002', '003', '004', '005']...


## 2. Understand MusingMusicId Type

The `MusingMusicId` is a dataclass that represents music identifiers (song IDs 1-12).

In [8]:
# Create music IDs for all 12 songs
song_ids = [MusingMusicId(song_id=i) for i in range(1, 13)]

print("Song IDs and their filenames:")
for music_id in song_ids:
    filename = music_id.to_filename()
    print(f"  Song {music_id.song_id}: {filename}")

# Alternative: Create MusicFilename directly from MusicID
music_filename = MusicFilename.from_musicid(MusingMusicId(song_id=1))
print(f"\nMusicFilename for song 1: {music_filename.filename}")

Song IDs and their filenames:
  Song 1: song_01.wav
  Song 2: song_02.wav
  Song 3: song_03.wav
  Song 4: song_04.wav
  Song 5: song_05.wav
  Song 6: song_06.wav
  Song 7: song_07.wav
  Song 8: song_08.wav
  Song 9: song_09.wav
  Song 10: song_10.wav
  Song 11: song_11.wav
  Song 12: song_12.wav

MusicFilename for song 1: song_01.wav


## 3. Load Subject Data

Load EEG data for specific subjects.

In [9]:
# Load data for the first subject
subject_id = "001"
subject_data = loader.load_subject_data(subject_id, max_sessions=3)  # Load first 3 songs only

print(f"\nLoaded data for subject {subject_id}:")
print(f"Sessions (songs): {list(subject_data.keys())}")

# Access session 1 (song 1)
session_1_data = subject_data['1']['1']  # session '1', run '1'
raw_eeg = session_1_data['raw']

print(f"\nSession 1 EEG data:")
print(f"  Sampling frequency: {raw_eeg.info['sfreq']} Hz")
print(f"  Number of channels: {len(raw_eeg.ch_names)}")
print(f"  Duration: {raw_eeg.n_times / raw_eeg.info['sfreq']:.2f} seconds")
print(f"  Channel names: {raw_eeg.ch_names[:5]}...")  # Show first 5 channels

Loading subject 001 (musin-g):


/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: Omitted 106 annotation(s) that were outside data range.
  raw = read_raw_bids(bids_path, verbose=False)
/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: Limited 2 annotation(s) that were expanding outside the data range.
  raw = read_raw_bids(bids_path, verbose=False)
/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = read_raw_bids(bids_path, verbose=False)
/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: Other is not an MNE-Python coordinate frame for EEG data and so will be set to 'unknown'
  raw = read_raw_bids(bids_path, verbose=False)
/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: participants.tsv file not found for datasets/m

  ✓ Song 01 (run 1): 136.0s, 128 events


/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: Omitted 110 annotation(s) that were outside data range.
  raw = read_raw_bids(bids_path, verbose=False)
/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = read_raw_bids(bids_path, verbose=False)
/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: Other is not an MNE-Python coordinate frame for EEG data and so will be set to 'unknown'
  raw = read_raw_bids(bids_path, verbose=False)
/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: participants.tsv file not found for datasets/musin_g_data/sub-001/ses-02/eeg/sub-001_ses-02_task-MusicListening_run-2_eeg.set
  raw = read_raw_bids(bids_path, verbose=False)


  ✓ Song 02 (run 2): 124.9s, 128 events


/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: Omitted 106 annotation(s) that were outside data range.
  raw = read_raw_bids(bids_path, verbose=False)
/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = read_raw_bids(bids_path, verbose=False)
/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = read_raw_bids(bids_path, verbose=False)
/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: Other is not an MNE-Python coordinate frame for EEG data and so will be set to 'unknown'
  raw = read_raw_bids(bids_path, verbose=False)


  ✓ Song 03 (run 3): 142.8s, 128 events

Loaded data for subject 001:
Sessions (songs): ['01', '02', '03']


/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: participants.tsv file not found for datasets/musin_g_data/sub-001/ses-03/eeg/sub-001_ses-03_task-MusicListening_run-3_eeg.set
  raw = read_raw_bids(bids_path, verbose=False)


KeyError: '1'

## 4. Get Dataset Statistics

In [10]:
# Load all subjects (or limited number)
loader.load_all_subjects(max_subjects=5, max_sessions_per_subject=3, verbose=True)

# Display statistics
loader.get_dataset_statistics()

🔄 Loading musin-g dataset...
📦 Subjects to load: 5 of 20
📅 Max sessions per subject: 3

Loading subject 001 (musin-g):


/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: Omitted 106 annotation(s) that were outside data range.
  raw = read_raw_bids(bids_path, verbose=False)
/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: Limited 2 annotation(s) that were expanding outside the data range.
  raw = read_raw_bids(bids_path, verbose=False)
/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = read_raw_bids(bids_path, verbose=False)
/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: Other is not an MNE-Python coordinate frame for EEG data and so will be set to 'unknown'
  raw = read_raw_bids(bids_path, verbose=False)
/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: participants.tsv file not found for datasets/m

  ✓ Song 01 (run 1): 136.0s, 128 events


/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: Omitted 110 annotation(s) that were outside data range.
  raw = read_raw_bids(bids_path, verbose=False)
/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = read_raw_bids(bids_path, verbose=False)
/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: Other is not an MNE-Python coordinate frame for EEG data and so will be set to 'unknown'
  raw = read_raw_bids(bids_path, verbose=False)
/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: participants.tsv file not found for datasets/musin_g_data/sub-001/ses-02/eeg/sub-001_ses-02_task-MusicListening_run-2_eeg.set
  raw = read_raw_bids(bids_path, verbose=False)


  ✓ Song 02 (run 2): 124.9s, 128 events


...

/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: Omitted 106 annotation(s) that were outside data range.
  raw = read_raw_bids(bids_path, verbose=False)
/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = read_raw_bids(bids_path, verbose=False)
/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = read_raw_bids(bids_path, verbose=False)
/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: Other is not an MNE-Python coordinate frame for EEG data and so will be set to 'unknown'
  raw = read_raw_bids(bids_path, verbose=False)
/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: participants.tsv file not found for datasets/m

  ✓ Song 01 (run 1): 136.0s, 128 events


/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: Omitted 110 annotation(s) that were outside data range.
  raw = read_raw_bids(bids_path, verbose=False)
/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = read_raw_bids(bids_path, verbose=False)
/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: Other is not an MNE-Python coordinate frame for EEG data and so will be set to 'unknown'
  raw = read_raw_bids(bids_path, verbose=False)
/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: participants.tsv file not found for datasets/musin_g_data/sub-005/ses-02/eeg/sub-005_ses-02_task-MusicListening_run-2_eeg.set
  raw = read_raw_bids(bids_path, verbose=False)


  ✓ Song 02 (run 2): 124.9s, 128 events
  ✓ Song 03 (run 3): 142.8s, 128 events

📊 LOADING SUMMARY:
• Successfully loaded: 5 subjects
• Failed to load: 0 subjects
• Success rate: 100.0%
📈 MUSIN-G DATASET STATISTICS:
• Loaded subjects: 5
• Total sessions: 15
• Total runs: 15
• Total trials: 15

🎵 EXPERIMENTAL PARADIGM:
• Type: Music Listening
• Trial structure: 12 complete songs, one per session
• Music type: Commercial music tracks (12 diverse genres)

⚙️  TECHNICAL SPECIFICATIONS:
• Sampling rate: 250.0 Hz
• Channels per recording: 129
• Average run duration: ~136 seconds


/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: Omitted 106 annotation(s) that were outside data range.
  raw = read_raw_bids(bids_path, verbose=False)
/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = read_raw_bids(bids_path, verbose=False)
/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = read_raw_bids(bids_path, verbose=False)
/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: Other is not an MNE-Python coordinate frame for EEG data and so will be set to 'unknown'
  raw = read_raw_bids(bids_path, verbose=False)
/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: participants.tsv file not found for datasets/m

## 5. Iterate Over Trials

The trial_iterator yields TrialRow objects with EEG data and music identifiers.

In [11]:
# Iterate over first 5 trials
print("\nFirst 5 trials:")
for i, trial in enumerate(loader.trial_iterator()):
    if i >= 5:
        break
    
    print(f"\nTrial {i+1}:")
    print(f"  Dataset: {trial.dataset}")
    print(f"  Subject: {trial.subject}")
    print(f"  Session (Song ID): {trial.session}")
    print(f"  Run: {trial.run}")
    print(f"  Trial ID: {trial.trial_id}")
    print(f"  Music filename: {trial.music_filename.filename}")
    print(f"  EEG duration: {trial.eeg_data.length_seconds():.2f}s")
    
    # Access the raw EEG data
    raw_eeg = trial.eeg_data.raw_eeg
    print(f"  EEG shape: {raw_eeg.get_data().shape}")


First 5 trials:
Reading 0 ... 34008  =      0.000 ...   136.032 secs...

Trial 1:
  Dataset: musin-g
  Subject: 001
  Session (Song ID): 01
  Run: 1
  Trial ID: song_01
  Music filename: song_01.wav
  EEG duration: 136.04s
  EEG shape: (129, 34009)
Reading 0 ... 31228  =      0.000 ...   124.912 secs...

Trial 2:
  Dataset: musin-g
  Subject: 001
  Session (Song ID): 02
  Run: 2
  Trial ID: song_02
  Music filename: song_02.wav
  EEG duration: 124.92s
  EEG shape: (129, 31229)
Reading 0 ... 35695  =      0.000 ...   142.780 secs...

Trial 3:
  Dataset: musin-g
  Subject: 001
  Session (Song ID): 03
  Run: 3
  Trial ID: song_03
  Music filename: song_03.wav
  EEG duration: 142.78s
  EEG shape: (129, 35696)
Reading 0 ... 136049  =      0.000 ...   136.049 secs...

Trial 4:
  Dataset: musin-g
  Subject: 002
  Session (Song ID): 01
  Run: 1
  Trial ID: song_01
  Music filename: song_01.wav
  EEG duration: 136.05s
  EEG shape: (129, 136050)
Reading 0 ... 124915  =      0.000 ...   124.915 

## 6. Export Dataset Using copy_from_dataloader_into_dir

We can use the `copy_from_dataloader_into_dir` function to save the dataset in a standardized format.

In [13]:
# Define output directory
output_dir = project_root / "datasets" / "musin_g_exported"

# Load limited data for export demo
loader_export = BCMIMusingLoader(str(musin_g_path))
loader_export.load_all_subjects(max_subjects=2, max_sessions_per_subject=2, verbose=False)

print(f"Exporting dataset to: {output_dir}")
print("This will create:")
print("  - eeg/ directory with EEG data organized by dataset/subject/session/run/trial_id")
print("  - stimuli/ directory (empty for musin-g as audio files not included)")
print("  - metadata.json with trial information")

# Uncomment to actually export:
# copy_from_dataloader_into_dir(loader_export, output_dir)
# print("Export complete!")

Loading subject 001 (musin-g):


/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: Omitted 106 annotation(s) that were outside data range.
  raw = read_raw_bids(bids_path, verbose=False)
/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: Limited 2 annotation(s) that were expanding outside the data range.
  raw = read_raw_bids(bids_path, verbose=False)
/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = read_raw_bids(bids_path, verbose=False)
/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: Other is not an MNE-Python coordinate frame for EEG data and so will be set to 'unknown'
  raw = read_raw_bids(bids_path, verbose=False)
/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: participants.tsv file not found for datasets/m

  ✓ Song 01 (run 1): 136.0s, 128 events


/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: Omitted 110 annotation(s) that were outside data range.
  raw = read_raw_bids(bids_path, verbose=False)
/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = read_raw_bids(bids_path, verbose=False)
/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: Other is not an MNE-Python coordinate frame for EEG data and so will be set to 'unknown'
  raw = read_raw_bids(bids_path, verbose=False)
/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: participants.tsv file not found for datasets/musin_g_data/sub-001/ses-02/eeg/sub-001_ses-02_task-MusicListening_run-2_eeg.set
  raw = read_raw_bids(bids_path, verbose=False)


  ✓ Song 02 (run 2): 124.9s, 128 events
Loading subject 002 (musin-g):


/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: Omitted 106 annotation(s) that were outside data range.
  raw = read_raw_bids(bids_path, verbose=False)
/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: Limited 1 annotation(s) that were expanding outside the data range.
  raw = read_raw_bids(bids_path, verbose=False)
/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = read_raw_bids(bids_path, verbose=False)
/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: Other is not an MNE-Python coordinate frame for EEG data and so will be set to 'unknown'
  raw = read_raw_bids(bids_path, verbose=False)
/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: participants.tsv file not found for datasets/m

  ✓ Song 01 (run 1): 136.0s, 128 events


/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: Omitted 110 annotation(s) that were outside data range.
  raw = read_raw_bids(bids_path, verbose=False)
/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = read_raw_bids(bids_path, verbose=False)
/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: Other is not an MNE-Python coordinate frame for EEG data and so will be set to 'unknown'
  raw = read_raw_bids(bids_path, verbose=False)


  ✓ Song 02 (run 2): 124.9s, 128 events
Exporting dataset to: /home/zmrocze/studia/uwr/eeg-magisterka/datasets/musin_g_exported
This will create:
  - eeg/ directory with EEG data organized by dataset/subject/session/run/trial_id
  - stimuli/ directory (empty for musin-g as audio files not included)
  - metadata.json with trial information


/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1521: RuntimeWarning: participants.tsv file not found for datasets/musin_g_data/sub-002/ses-02/eeg/sub-002_ses-02_task-MusicListening_run-2_eeg.set
  raw = read_raw_bids(bids_path, verbose=False)


In [ ]:
loader_export = BCMIMusingLoader(str(musin_g_path))
loader_export.load_all_subjects()
new_dataset_save_dir = Path("./datasets/musin_g_export")
copy_from_dataloader_into_dir(loader_export, new_dataset_save_dir)

ds = EEGMusicDataset.load_ondisk(new_dataset_save_dir)

Loading subject 001 (musin-g):


/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1524: RuntimeWarning: Omitted 106 annotation(s) that were outside data range.
  raw = read_raw_bids(bids_path, verbose=False)
/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1524: RuntimeWarning: Limited 2 annotation(s) that were expanding outside the data range.
  raw = read_raw_bids(bids_path, verbose=False)
/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1524: RuntimeWarning: The data contains 'boundary' events, indicating data discontinuities. Be cautious of filtering and epoching around these events.
  raw = read_raw_bids(bids_path, verbose=False)
/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1524: RuntimeWarning: Other is not an MNE-Python coordinate frame for EEG data and so will be set to 'unknown'
  raw = read_raw_bids(bids_path, verbose=False)
/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1524: RuntimeWarning: participants.tsv file not found for datasets/m

  ✓ Song 01 (run 1): 136.0s, 128 events


/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1524: RuntimeWarning: Other is not an MNE-Python coordinate frame for EEG data and so will be set to 'unknown'
  raw = read_raw_bids(bids_path, verbose=False)
/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/bcmi.py:1524: RuntimeWarning: participants.tsv file not found for datasets/musin_g_data/sub-001/ses-02/eeg/sub-001_ses-02_task-MusicListening_run-2_eeg.set
  raw = read_raw_bids(bids_path, verbose=False)


  ✓ Song 02 (run 2): 124.9s, 128 events
Loading subject 002 (musin-g):


...

Reading 0 ... 136049  =      0.000 ...   136.049 secs...
Overwriting existing file.


/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/data.py:496: RuntimeWarning: EDF format requires equal-length data blocks, so 0.95 seconds of edge values were appended to all channels when writing the final block.
  mne.export.export_raw(filepath, self.raw_eeg, fmt="edf", overwrite=True)


Reading 0 ... 124915  =      0.000 ...   124.915 secs...
Overwriting existing file.


/home/zmrocze/studia/uwr/eeg-magisterka/src/eeg_music/data.py:496: RuntimeWarning: EDF format requires equal-length data blocks, so 0.084 seconds of edge values were appended to all channels when writing the final block.
  mne.export.export_raw(filepath, self.raw_eeg, fmt="edf", overwrite=True)


ValueError: Index has duplicate keys: MultiIndex([('musin-g', '001', '01',  '1', 'song_01'),
            ('musin-g', '001', '02',  '2', 'song_02'),
            ('musin-g', '001', '03',  '3', 'song_03'),
            ('musin-g', '001', '04',  '4', 'song_04'),
            ('musin-g', '001', '05',  '5', 'song_05'),
            ('musin-g', '001', '06',  '6', 'song_06'),
            ('musin-g', '001', '07',  '7', 'song_07'),
            ('musin-g', '001', '08',  '8', 'song_08'),
            ('musin-g', '001', '09',  '9', 'song_09'),
            ('musin-g', '001', '10', '10', 'song_10'),
            ...
            ('musin-g', '020', '03',  '3', 'song_03'),
            ('musin-g', '020', '04',  '4', 'song_04'),
            ('musin-g', '020', '05',  '5', 'song_05'),
            ('musin-g', '020', '06',  '6', 'song_06'),
            ('musin-g', '020', '07',  '7', 'song_07'),
            ('musin-g', '020', '08',  '8', 'song_08'),
            ('musin-g', '020', '09',  '9', 'song_09'),
           

In [4]:
ds

NameError: name 'ds' is not defined

## 7. Load Exported Dataset with EegMusicDataset

After exporting, we can load the dataset using EegMusicDataset.

In [ ]:
# Assuming you've exported the dataset in step 6
# exported_path = output_dir

# Load with EegMusicDataset
# dataset = EegMusicDataset(base_dir=exported_path)
# print(f"Loaded {len(dataset)} trials")

# Access a trial
# trial = dataset[0]
# print(f"\nFirst trial:")
# print(trial.pretty())

## 8. Access Behavioral Data

The MUSIN-G dataset includes behavioral ratings (enjoyment and familiarity) in the stimuli directory.

In [ ]:
import pandas as pd

# Load behavioral data
behavioral_file = musin_g_path / "stimuli" / "Behavioural_data"
behavioral_df = pd.read_csv(behavioral_file, sep="\t")

print("Behavioral data columns:", behavioral_df.columns.tolist())
print(f"\nShape: {behavioral_df.shape}")
print("\nFirst few rows:")
print(behavioral_df.head(10))

# Get ratings for subject 1, song 1
subject_1_song_1 = behavioral_df[(behavioral_df['Subject'] == 1) & (behavioral_df['Song_ID'] == 1)]
if not subject_1_song_1.empty:
    enjoyment = subject_1_song_1['Enjoyment'].iloc[0]
    familiarity = subject_1_song_1['Familiarity'].iloc[0]
    print(f"\nSubject 1, Song 1:")
    print(f"  Enjoyment: {enjoyment}/5")
    print(f"  Familiarity: {familiarity}/5")

## 9. Song Information

The musing.py module contains detailed information about all 12 songs.

In [ ]:
from eeg_music.musing import songs_info_enhanced

print("\nSong Information:")
print("=" * 80)
for song_id, info in songs_info_enhanced.items():
    print(f"\nSong {song_id}: {info['name']}")
    print(f"  Artist: {info['artist']}")
    print(f"  Genre: {info['genre']}")
    print(f"  Duration: {info['duration']}s")
    print(f"  Tempo: {info['tempo']} BPM" if info['tempo'] else "  Tempo: N/A")
    print(f"  Characteristics: {info['characteristics']}")

## Summary

This notebook demonstrated:

1. **Loading MUSIN-G dataset** with `BCMIMusingLoader`
2. **Understanding `MusingMusicId`** type (integers 1-12 for 12 songs)
3. **Accessing EEG data** for subjects and sessions
4. **Iterating over trials** with `trial_iterator()`
5. **Exporting dataset** with `copy_from_dataloader_into_dir()`
6. **Loading exported data** with `EegMusicDataset`
7. **Accessing behavioral ratings** (enjoyment and familiarity)
8. **Song metadata** from the musing module

### Key Features of MUSIN-G Loader:

- **Compatible with BCMI framework**: Extends `BaseBCMILoader`
- **BIDS format support**: Uses mne_bids for loading
- **Session-based organization**: Each of 12 sessions represents one song
- **Full song trials**: Each trial contains the complete EEG recording for one song
- **No audio files**: The dataset contains only EEG data and metadata
- **Behavioral data**: Ratings available separately in stimuli directory